# Pegasus XSum

## Introduction
This notebook demonstrates **Pegasus XSum** for text summarization. This pretrained checkpoint generates a domain-adapted abstractive summary.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [1]:
import os, sys
from pathlib import Path
CURRENT_DIR = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (candidate / "summarization_utils.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing summarization_utils.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


# Single Document Pegasus
**Pegasus** is a state-of-the-art language model developed by Google that has shown impressive results in several natural language processing tasks, including text summarization. Here are some pros and cons of using Pegasus for single document summarization:

## Pros:

* High-quality summaries: Pegasus is known for generating high-quality summaries that capture the main ideas and key points of the input document accurately. This is because Pegasus is trained on a large corpus of diverse texts, which helps it to understand the nuances of language better.

* Generalization: Pegasus can generalize to unseen data, which means that it can summarize documents on topics it has not encountered during training. This makes it suitable for summarizing a wide range of texts, including those outside its training domain.

* Abstractive summarization: Pegasus uses abstractive summarization, which means that it can generate summaries that are not just a selection of sentences from the input document, but rather a rephrasing of the content in a more concise and coherent manner. This makes Pegasus summaries more readable and engaging.

* Speed: Pegasus is relatively fast at generating summaries, which makes it suitable for real-time applications, such as news summarization.

## Cons:

* Large computational requirements: Pegasus requires significant computational resources, including a powerful GPU and a large amount of memory. This can make it challenging to run on low-end hardware or in resource-constrained environments.

* Limited control: Pegasus generates summaries automatically, without any input from the user. This means that users have limited control over the length and content of the summary.

* Limited transparency: Because Pegasus is a black-box model, it can be challenging to understand how it generates summaries. This can make it difficult to diagnose errors or to fine-tune the model for specific tasks.

* Domain-specific summarization: Pegasus is trained on a diverse corpus of texts, but it may not perform as well on documents that are highly specialized or technical in nature. In such cases, a domain-specific summarization model may be more appropriate.

These are the scores we achieved:

    ROUGE Score:
    Precision: 0.583
    Recall: 0.046
    F1-Score: 0.086

    BLEU Score: 0

### References

Here are some related papers on Pegasus:

1. "PEGASUS: Pre-training with Extracted Gap-sentences for Abstractive Summarization" by Jingqing Zhang, Yao Zhao, Mohammad Saleh, and Peter J. Liu. This is the original paper that introduced Pegasus and describes the pre-training method used to train the model.

1. "Fine-Tuning Pretrained Language Models: Weight Initializations, Data Orders, and Early Stopping" by Yanyao Shen, Xiaodong Liu, Kevin Duh, and Jianfeng Gao. This paper explores various fine-tuning strategies for Pegasus and other language models, including weight initialization, data ordering, and early stopping.

1. "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer" by Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, and Peter J. Liu. This paper presents a unified text-to-text transformer model, which includes Pegasus, and evaluates its performance on a range of natural language processing tasks.

1. "Text Summarization with Pretrained Encoders" by Yang Liu and Mirella Lapata. This paper explores the use of pretrained language models, including Pegasus, for text summarization and compares their performance to traditional summarization models.

1. "Beyond Accuracy: Behavioral Testing of NLP Models with CheckList" by Marco Tulio Ribeiro, Tongshuang Wu, Carlos Guestrin, and Sameer Singh. This paper presents CheckList, a suite of behavioral tests for natural language processing models, including Pegasus, to evaluate their performance on specific linguistic phenomena.





**Pegasus-XSUM** is a pre-trained language model that was introduced in 2020 for the task of extreme summarization. It is based on the Pegasus architecture, which uses a transformer-based sequence-to-sequence model. Pegasus-XSUM was trained on a large dataset of document-summary pairs called XSUM, which contains over 220,000 examples. The model has achieved state-of-the-art performance on the XSUM dataset and has shown promising results on other summarization tasks. Pegasus-XSUM is available as part of the Hugging Face Transformers library, making it easy to use for natural language processing tasks.

In [2]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from summarization_utils import load_article, load_reference, evaluate_summary, format_scores

    article = load_article()
    reference_summary = load_reference()
    model_name = 'google/pegasus-xsum'
    # The MultiNews checkpoint expects document separators; all other checkpoints receive the article as one document.
    model_input = article
    if model_name.endswith("multi_news"):
        model_input = " ||| ".join(part.strip() for part in article.split("\n\n") if part.strip())

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    max_input_tokens = max(1, min(1024, getattr(model.config, "max_position_embeddings", 1024) - 2))
    inputs = tokenizer(model_input, truncation=True, max_length=max_input_tokens, return_tensors="pt")
    with torch.inference_mode():
        generated = model.generate(**inputs, num_beams=4, max_length=120, min_length=20, no_repeat_ngram_size=3)
    summary = tokenizer.decode(generated[0], skip_special_tokens=True)
    print("Summary:", summary)
    print("Metrics:", format_scores(evaluate_summary(summary, reference_summary)))
except (OSError, RuntimeError, ImportError) as error:
    summary = None
    print(f"Pegasus inference unavailable in this environment: {type(error).__name__}: {error}")


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Summary: Global temperatures in 2025 were the third warmest on record, according to the US National Oceanic and Atmospheric Administration (NOAA).
Metrics: rouge1_f1=0.202, rouge2_f1=0.046, rougeL_f1=0.157, bleu4=0.012
